<a href="https://colab.research.google.com/github/KeerthanaSistla/Compiler-Design/blob/main/Compiler_Design_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re

def convert_line(line):
    line = line.strip()

    # Convert print
    if line.startswith("print("):
        content = line[6:-1]
        return f'System.out.println({content});'

    # Convert variable assignment (basic)
    match = re.match(r"(\w+)\s*=\s*(.+)", line)
    if match:
        var, value = match.groups()
        return f"var {var} = {value};"

    # Convert if statement
    if line.startswith("if "):
        condition = line[3:].rstrip(":")
        return f"if ({condition}) {{"

    # Convert elif → else if
    if line.startswith("elif "):
        condition = line[5:].rstrip(":")
        return f"}} else if ({condition}) {{"

    # Convert else
    if line.startswith("else"):
        return "} else {"

    # Convert for loop (range only)
    match = re.match(r"for (\w+) in range\((\d+)\):", line)
    if match:
        var, end = match.groups()
        return f"for (int {var} = 0; {var} < {end}; {var}++) {{"

    # Convert function
    match = re.match(r"def (\w+)\((.*?)\):", line)
    if match:
        name, params = match.groups()
        params = params.replace(",", ", Object ")
        if params:
            params = "Object " + params
        return f"public static void {name}({params}) {{"

    return line


def convert_python_to_java(python_code):
    java_code = []
    indent_stack = []

    for line in python_code.split("\n"):
        stripped = line.strip()

        if not stripped:
            continue

        converted = convert_line(line)
        java_code.append(converted)

        # Close blocks (very naive)
        if stripped.endswith(":"):
            indent_stack.append("}")

    # Close remaining blocks
    while indent_stack:
        java_code.append(indent_stack.pop())

    return "\n".join(java_code)


# Example usage
python_code = """
x = 5
print(x)

if x > 3:
    print("Big")
else:
    print("Small")

for i in range(5):
    print(i)
"""

java_output = convert_python_to_java(python_code)

print("=== Java Code ===")
print(java_output)

=== Java Code ===
var x = 5;
System.out.println(x);
if (x > 3) {
System.out.println("Big");
} else {
System.out.println("Small");
for (int i = 0; i < 5; i++) {
System.out.println(i);
}
}
}


In [1]:
import re

def get_indent_level(line):
    return len(line) - len(line.lstrip(" "))


def convert_line(line):
    stripped = line.strip()

    if stripped.startswith("print("):
        return f"System.out.println({stripped[6:-1]});"

    match = re.match(r"(\w+)\s*=\s*(.+)", stripped)
    if match:
        var, value = match.groups()
        return f"var {var} = {value};"

    if stripped.startswith("if "):
        return f"if ({stripped[3:].rstrip(':')}) {{"

    if stripped.startswith("elif "):
        return f"}} else if ({stripped[5:].rstrip(':')}) {{"

    if stripped.startswith("else"):
        return "} else {"

    match = re.match(r"for (\w+) in range\((\d+)\):", stripped)
    if match:
        var, end = match.groups()
        return f"for (int {var} = 0; {var} < {end}; {var}++) {{"

    return stripped


def convert_python_to_java(python_code):
    lines = python_code.split("\n")
    java_lines = []
    indent_stack = []

    for line in lines:
        if not line.strip():
            continue

        indent = get_indent_level(line)
        stripped = line.strip()

        # Close blocks when indentation decreases
        while indent_stack and indent < indent_stack[-1]:
            java_lines.append(" " * (indent_stack[-1] + 8) + "}")
            indent_stack.pop()

        converted = convert_line(line)
        java_lines.append(" " * (indent + 8) + converted)

        # Only push for real new blocks (NOT else/elif)
        if stripped.endswith(":") and not stripped.startswith(("else", "elif")):
            indent_stack.append(indent)

    # Close remaining blocks
    while indent_stack:
        java_lines.append(" " * (indent_stack[-1] + 8) + "}")
        indent_stack.pop()

    return "\n".join([
        "class Main {",
        "    public static void main(String[] args) {",
        *java_lines,
        "    }",
        "}"
    ])


# Test
python_code = """
x = 10
print(x)

if x > 5:
    print("Big")
else:
    print("Small")
"""

print(convert_python_to_java(python_code))

class Main {
    public static void main(String[] args) {
        var x = 10;
        System.out.println(x);
        if (x > 5) {
            System.out.println("Big");
        } else {
            System.out.println("Small");
        }
    }
}


In [3]:
import ast

class PyToJavaCompiler(ast.NodeVisitor):
    def __init__(self):
        self.java = []
        self.indent = 0

    def emit(self, line):
        self.java.append("    " * self.indent + line)

    # ---------- Program ----------
    def visit_Module(self, node):
        self.emit("import java.util.*;")
        self.emit("")
        self.emit("class Main {")
        self.indent += 1

        # main method
        self.emit("public static void main(String[] args) {")
        self.indent += 1

        for stmt in node.body:
            if not isinstance(stmt, ast.FunctionDef):
                self.visit(stmt)

        self.indent -= 1
        self.emit("}")
        self.emit("")

        # functions
        for stmt in node.body:
            if isinstance(stmt, ast.FunctionDef):
                self.visit(stmt)

        self.indent -= 1
        self.emit("}")

    # ---------- Function ----------
    def visit_FunctionDef(self, node):
        params = []
        for arg in node.args.args:
            if arg.arg == "arr":
                params.append("int[] " + arg.arg)
            else:
                params.append("int " + arg.arg)

        self.emit(f"static void {node.name}({', '.join(params)}) {{")
        self.indent += 1

        for stmt in node.body:
            self.visit(stmt)

        self.indent -= 1
        self.emit("}")
        self.emit("")

    # ---------- Assign ----------
    def visit_Assign(self, node):
        target = node.targets[0].id

        if isinstance(node.value, ast.List):
            values = [self.visit(v) for v in node.value.elts]
            self.emit(f"int[] {target} = {{{', '.join(values)}}};")
        else:
            value = self.visit(node.value)
            self.emit(f"int {target} = {value};")

    # ---------- If ----------
    def visit_If(self, node):
        cond = self.visit(node.test)
        self.emit(f"if ({cond}) {{")
        self.indent += 1

        for stmt in node.body:
            self.visit(stmt)

        self.indent -= 1

        if node.orelse:
            self.emit("} else {")
            self.indent += 1
            for stmt in node.orelse:
                self.visit(stmt)
            self.indent -= 1

        self.emit("}")

    # ---------- While ----------
    def visit_While(self, node):
        cond = self.visit(node.test)
        self.emit(f"while ({cond}) {{")
        self.indent += 1

        for stmt in node.body:
            self.visit(stmt)

        self.indent -= 1
        self.emit("}")

    # ---------- For ----------
    def visit_For(self, node):
        if isinstance(node.iter, ast.Call) and node.iter.func.id == "range":
            args = node.iter.args
            var = node.target.id

            if len(args) == 1:
                end = self.visit(args[0])
                self.emit(f"for (int {var} = 0; {var} < {end}; {var}++) {{")

            elif len(args) == 2:
                start = self.visit(args[0])
                end = self.visit(args[1])
                self.emit(f"for (int {var} = {start}; {var} < {end}; {var}++) {{")

            self.indent += 1
            for stmt in node.body:
                self.visit(stmt)
            self.indent -= 1
            self.emit("}")

    # ---------- Expressions ----------
    def visit_Expr(self, node):
        self.emit(self.visit(node.value) + ";")

    def visit_Call(self, node):
        if isinstance(node.func, ast.Name):

            # print
            if node.func.id == "print":
                args = ", ".join(self.visit(a) for a in node.args)
                return f"System.out.println({args})"

            # function calls
            return f"{node.func.id}({', '.join(self.visit(a) for a in node.args)})"

        return ""

    # ---------- Binary ----------
    def visit_BinOp(self, node):
        left = self.visit(node.left)
        right = self.visit(node.right)

        if isinstance(node.op, ast.FloorDiv):
            return f"({left} / {right})"

        return f"({left} {self.op(node.op)} {right})"

    def op(self, op):
        return {
            ast.Add: "+",
            ast.Sub: "-",
            ast.Mult: "*",
            ast.Div: "/",
            ast.Mod: "%"
        }[type(op)]

    # ---------- Compare ----------
    def visit_Compare(self, node):
        left = self.visit(node.left)
        right = self.visit(node.comparators[0])
        op = {
            ast.Lt: "<",
            ast.Gt: ">",
            ast.LtE: "<=",
            ast.GtE: ">=",
            ast.Eq: "=="
        }[type(node.ops[0])]
        return f"{left} {op} {right}"

    # ---------- Name / Constant ----------
    def visit_Name(self, node):
        return node.id

    def visit_Constant(self, node):
        return str(node.value)

    # ---------- Subscript ----------
    def visit_Subscript(self, node):
        return f"{self.visit(node.value)}[{self.visit(node.slice)}]"

    def visit_Index(self, node):
        return self.visit(node.value)


# ================= RUN =================

python_code = """
def merge_sort(arr, left, right):
    if left < right:
        mid = (left + right) // 2
"""

tree = ast.parse(python_code)
compiler = PyToJavaCompiler()
compiler.visit(tree)

print("\n".join(compiler.java))

import java.util.*;

class Main {
    public static void main(String[] args) {
    }
    
    static void merge_sort(int[] arr, int left, int right) {
        if (left < right) {
            int mid = ((left + right) / 2);
        }
    }
    
}
